In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import proplot as pplt
from pathlib import Path
import warnings

from AOSCMcoupling.convergence_checker import relative_error

In [ ]:
class OASISPreprocessor:
    def __init__(
        self, origin: pd.Timestamp = None, time_shift: pd.Timedelta = pd.Timedelta(0)
    ):
        """Constructor.

        :param origin: start date (+ time) of the simulation
        :type origin: pd.Timestamp
        :param time_shift: time shift to apply for local time, default: 0
        :type time_shift: pd.Timedelta, optional
        """
        self.origin = origin
        self.time_shift = time_shift

    def preprocess(self, ds: xr.Dataset) -> xr.DataArray:
        """Preprocess function for use with `xr.open_mfdataset()`.

        :param ds: dataset as loaded from disk
        :type ds: xr.Dataset
        :return: preprocessed DataArray (only one variable!)
        :rtype: xr.DataArray
        """
        ds = ds.isel(ny=0, nx=0)

        source_file = Path(ds.encoding["source"])
        iteration = int(source_file.parent.name.split("_")[-1])
        ds = ds.expand_dims(
            iteration=[iteration],
        )
        time_data = np.array(ds.time.data, dtype="timedelta64[s]")
        ds = ds.assign_coords(time=ds.time - ds.time[0])

        if self.origin is not None:
            time_data = np.array(ds.time.data, dtype="timedelta64[s]")
            ds = ds.assign_coords(time=self.origin + time_data + self.time_shift)
        return ds

In [ ]:
preprocessor = OASISPreprocessor(pd.Timestamp("2014-07-01"), pd.Timedelta(-7, "h"))

In [ ]:
coupling_vars = [
    "A_TauX_oce",
    "A_TauY_oce",
    "A_TauX_ice",
    "A_TauY_ice",
    "A_Qs_mix",
    "A_Qns_mix",
    "A_Qs_ice",
    "A_Qns_ice",
    "A_Precip_liquid",
    "A_Precip_solid",
    "A_Evap_total",
    "A_Evap_ice",
    "A_dQns_dT",
    "O_SSTSST",
    "O_TepIce",
    "O_AlbIce",
    "OIceFrc",
    "OIceTck",
    "OSnwTck",
]

In [ ]:
exp_ids = ["T1HB", "T1HC", "T12H", "T2DA"]
titles = [r"$\Delta t_\mathrm{cpl} = 1h, \mathcal{T}=1h$", r"$\Delta t_\mathrm{cpl} = 1h, \mathcal{T}=12h$", r"$\Delta t_\mathrm{cpl} = 12h, \mathcal{T}=12h$", r"$\Delta t_\mathrm{cpl} = 1h, \mathcal{T}=48h$", ]

fig, axs = pplt.subplots(width="95em", height="30em", ncols=4)
for i, exp_id in enumerate(exp_ids):
    ax = axs[i]
    output_dir = Path("PAPA")
    netcdf_files = [file for file in output_dir.glob(f"{exp_id}_*/*.nc")]
    oasis_files = [
        file for file in netcdf_files if any(cv in file.name for cv in coupling_vars)
    ]

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        ds = xr.open_mfdataset(oasis_files, preprocess=preprocessor.preprocess)
    rel_errors_inf = relative_error(
        ds, ds.isel(iteration=-1), ds.isel(iteration=0), ord=np.inf
    )

    rel_errors_inf.sel(iteration=np.arange(1, 20)).O_SSTSST.plot(
        ax=ax, ls="-", marker=".", label="SST", color="k"
    )
    rel_errors_inf.sel(iteration=np.arange(1, 20)).A_Qns_mix.plot(
        ax=ax, ls="--", marker=".", label=r"$Q_\mathrm{ns}$", color="k"
    )
    rel_errors_inf.sel(iteration=np.arange(1, 20)).A_TauX_oce.plot(
        ax=ax, ls=":", marker="x", label=r"$\tau_x$", color="k"
    )
    rel_errors_inf.sel(iteration=np.arange(1, 20)).A_Precip_liquid.plot(
        ax=ax, ls="-.", marker="1", label=r"$\mathcal{P}$", color="k"
    )

    ax.format(
        yscale="log",
        yformatter="sci",
        ylabel="Relative Error",
        xlabel="Iteration",
        ylim=[1e-14, 1e-1],
        title=titles[i],
    )
    ax.legend(ncols=1)

    if i == 3:
        factor_Qns = np.sqrt(rel_errors_inf.A_Qns_mix.sel(iteration=np.arange(10, 16)) / rel_errors_inf.A_Qns_mix.sel(iteration=np.arange(8, 14)).data)
    else:
        factor_Qns = np.sqrt(rel_errors_inf.A_Qns_mix.sel(iteration=np.arange(5, 11)) / rel_errors_inf.A_Qns_mix.sel(iteration=np.arange(3, 9)).data)
    print(f"{exp_id}: {float(np.mean(factor_Qns))}")
# fig.savefig("cex_swr_convergence_combined.pdf")

In [ ]:
preprocessor = OASISPreprocessor()
exp_ids = ["TOP1", "TOPR"]
titles = [r"$\Delta t_\mathrm{cpl} = 1h, \mathcal{T}=1h$", r"$\Delta t_\mathrm{cpl} = 1h, \mathcal{T}=48h$", ]

fig, axs = pplt.subplots(width="60em", height="30em", ncols=2)
output_dir = Path("output")
for i, exp_id in enumerate(exp_ids):
    ax = axs[i]
    netcdf_files = [file for file in output_dir.glob(f"{exp_id}_*/*.nc")]
    oasis_files = [
        file for file in netcdf_files if any(cv in file.name for cv in coupling_vars)
    ]

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        ds = xr.open_mfdataset(oasis_files, preprocess=preprocessor.preprocess)
    rel_errors_inf = relative_error(
        ds, ds.isel(iteration=-1), ds.isel(iteration=0), ord=np.inf
    )

    rel_errors_inf.sel(iteration=np.arange(1, 30)).O_SSTSST.plot(
        ax=ax, ls="-", marker=".", label="SST", color="k"
    )
    rel_errors_inf.sel(iteration=np.arange(1, 30)).A_Qns_mix.plot(
        ax=ax, ls="--", marker=".", label=r"$Q_\mathrm{ns}$", color="k"
    )
    rel_errors_inf.sel(iteration=np.arange(1, 30)).A_TauX_oce.plot(
        ax=ax, ls=":", marker="x", label=r"$\tau_x$", color="k"
    )
    rel_errors_inf.sel(iteration=np.arange(1, 30)).A_Precip_liquid.plot(
        ax=ax, ls="-.", marker="1", label=r"$\mathcal{P}$", color="k"
    )

    ax.format(
        yscale="log",
        yformatter="sci",
        ylabel="Relative Error",
        xlabel="Iteration",
        ylim=[1e-14, 1e-1],
        title=titles[i],
    )
    ax.legend(ncols=1)

    if i == 0:
        factor_Qns = np.sqrt(rel_errors_inf.A_Qns_mix.sel(iteration=np.arange(3, 6)) / rel_errors_inf.A_Qns_mix.sel(iteration=np.arange(1, 4)).data)
    else:
        factor_Qns = np.sqrt(rel_errors_inf.A_Qns_mix.sel(iteration=np.arange(7, 13)) / rel_errors_inf.A_Qns_mix.sel(iteration=np.arange(5, 11)).data)
    print(f"{exp_id}: {float(np.mean(factor_Qns))}")

In [ ]:
print(float(ds.OIceFrc.max()))
print(float(ds.OIceFrc.min()))
print(float(ds.OIceTck.min()))
print(float(ds.OIceTck.max()))